# Jensen-Shannon Divergence

**Audience:** students who have seen KL divergence and Shannon entropy.

**What you will build:** a symmetric, bounded alternative to KL; numerical
examples from the lecture notes; a plot comparing KL and JS; and the identity
linking JS to mutual information.

## Learning objectives

1. compute $D_{\mathrm{JS}}(p\|q)$ from two distributions;
2. verify symmetry, non-negativity and the upper bound $\log 2$ for bits;
3. explain why JS remains finite when supports are disjoint;
4. interpret JS as the information gained about which distribution generated a sample.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update(
    {
        "figure.figsize": (8, 5),
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
print("Setup complete")

## 1. Definition

Let $m=\tfrac12(p+q)$ be the midpoint mixture. Then
$$D_{\mathrm{JS}}(p\,\|\,q) = \tfrac12 D_{\mathrm{KL}}(p\,\|\,m)
+ \tfrac12 D_{\mathrm{KL}}(q\,\|\,m).$$
Because $m$ is positive wherever $p$ or $q$ is positive, the KL terms are
always finite. The base-2 version lies in $[0,1]$ bit for binary laws.

In [ ]:
def entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))


def kl(p, q):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    support = p > 0
    return float(np.sum(p[support] * np.log2(p[support] / q[support])))


def js(p, q):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    m = 0.5 * (p + q)
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


print("helpers ready")

## 2. Concrete examples

We reproduce the three cases from the lecture: identical laws, disjoint
supports, and overlapping Bernoulli laws.

In [ ]:
cases = {
    "identical": ((0.5, 0.5), (0.5, 0.5)),
    "disjoint": ((1.0, 0.0), (0.0, 1.0)),
    "overlapping": ((0.9, 0.1), (0.1, 0.9)),
}
for name, (p, q) in cases.items():
    midpoint = 0.5 * (np.array(p) + np.array(q))
    print(f"{name:12s} JS = {js(p, q):.4f} bits, H(m) = {entropy(midpoint):.4f}")
print("\nsymmetry check:", js([0.9, 0.1], [0.1, 0.9]) == js([0.1, 0.9], [0.9, 0.1]))

## 3. KL versus JS

Fix $p=(0.9,0.1)$ and vary $q=(q_1,1-q_1)$. KL blows up as $q_1\to0$, while
JS stays bounded and symmetric around $q_1=0.5$.

In [ ]:
p = np.array([0.9, 0.1])
qs = np.linspace(1e-6, 1 - 1e-6, 300)
kl_values = [kl(p, [q, 1 - q]) for q in qs]
js_values = [js(p, [q, 1 - q]) for q in qs]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(qs, kl_values, color="tab:red")
axes[0].set(title="KL is unbounded", xlabel="q1", ylabel="bits")
axes[1].plot(qs, js_values, color="tab:blue")
axes[1].axhline(1.0, color="0.5", ls="--", label="maximum = 1 bit")
axes[1].set(title="JS is bounded", xlabel="q1", ylabel="bits")
axes[1].legend()
fig.tight_layout()
plt.show()

## 4. JS as mutual information

Draw a label $Z\in\{0,1\}$ with probability $1/2$; if $Z=0$ draw
$X\sim p$, otherwise $X\sim q$. Then
$I(Z;X)=D_{\mathrm{JS}}(p\|q)$. Knowing $X$ tells you which source produced it,
and the information is zero only when the two sources are identical.

In [ ]:
def js_as_mi(p, q):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    m = 0.5 * (p + q)
    total = 0.0
    for source in (p, q):
        joint = 0.5 * source
        valid = joint > 0
        total += np.sum(joint[valid] * np.log2(joint[valid] / (0.5 * m[valid])))
    return float(total)


for p, q in cases.values():
    print(f"JS={js(p, q):.6f}, MI={js_as_mi(p, q):.6f}")

## Exercises

1. Prove that JS is symmetric directly from the definition.
2. Show that JS is bounded above by 1 bit for binary distributions.
3. Verify that `np.sqrt(js(p, q))` satisfies the triangle inequality for several triples.
4. Compare JS and total variation distance for the overlapping Bernoulli case.
5. Use `Code/information_theory.py` to check your answers with a different implementation.